In [1]:
# load necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
print("--- Step 1: Loading Real Dataset & Auditing Quality ---")

--- Step 1: Loading Real Dataset & Auditing Quality ---


In [3]:
#import the csv file stored in the same folder
data = pd.read_csv('credit_risk_dataset.csv')

In [4]:
# preview the top 5 rows of the dataset
data.head(5)

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [5]:
# check for missing values in the dataset 

missing_values = data.isnull().sum()

print(f"\nMissing values in each column:\n{missing_values[missing_values > 0]}")


Missing values in each column:
person_emp_length     895
loan_int_rate        3116
dtype: int64


In [6]:
# Will fill missing employee length with the median value of the column
median_employee_length = data['person_emp_length'].median()
# print out the exact median value and then add it in 
print(f"\nMedian value of 'person_emp_length': {median_employee_length}")


Median value of 'person_emp_length': 4.0


In [7]:
# add the median value to the missing values in the 'person_emp_length' column
data['person_emp_length'] = data['person_emp_length'].fillna(median_employee_length)

In [8]:
#check if it worked by checking for missing values again
missing_values_after_fill = data.isnull().sum()

print(f"\nMissing values after filling 'person_emp_length':\n{missing_values_after_fill[missing_values_after_fill > 0]}")


Missing values after filling 'person_emp_length':
loan_int_rate    3116
dtype: int64


In [9]:
# Will also fill missing interest rates with median value
median_interest_rate = data['loan_int_rate'].median()
# Fill in missing columns with the new median value
data["loan_int_rate"] = data["loan_int_rate"].fillna(median_interest_rate)

In [10]:
# check to make sure that the missing values have been filled
missing_values_after_fill_interest = data.isnull().sum()

print(f"\nMissing values after filling 'loan_int_rate':\n{missing_values_after_fill_interest[missing_values_after_fill_interest > 0]}")


Missing values after filling 'loan_int_rate':
Series([], dtype: int64)


In [11]:
# Data quality check: one-hot encoding categorical variables to prepare for machine learning model
data_encoding_issues = pd.get_dummies(data, columns=['person_home_ownership', 'loan_intent', 'cb_person_default_on_file', 'loan_grade'], drop_first=True)

In [12]:
print("\n--- Step 2: Preparing and Splitting Data ---")


--- Step 2: Preparing and Splitting Data ---


In [13]:
# Target variable is 'loan_status', which indicates whether the loan was paid off or not. We will encode it as a binary variable (0 for paid off, 1 for defaulted).
x = data_encoding_issues.drop(columns=['loan_status'])
y = data_encoding_issues['loan_status']

In [14]:
# Split data (80% training, 20% testing) ensuring class distribution is maintained using stratification
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# PHASE 3: TECHNICAL EXECUTION (Scikit-Learn Random Forest)

In [15]:
print("\n--- Step 3: Engineering the Scikit-Learn Model ---")


--- Step 3: Engineering the Scikit-Learn Model ---


In [16]:
# Keep max_depth to 5 to prevent overfitting and set random_state for reproducibility
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
print('Model training completed.')

Model training completed.


In [17]:
print("\n--- Step 4: Extracting Business Intelligence & Explainability ---")


--- Step 4: Extracting Business Intelligence & Explainability ---


In [18]:
# Model predictions on the test set
y_pred = model.predict(X_test)
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


Confusion Matrix:
[[5060   35]
 [ 603  819]]


In [19]:
# Feature Importance Extraction
feature_importances = model.feature_importances_
feature_summary = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importances}).sort_values(by='Importance', ascending=False)

In [20]:
print("\n[EXECUTIVE INSIGHT] Core Drivers of Credit Defaults for Business Teams:")
for index, row in feature_summary.head(4).iterrows():
    clean_feature_name = row['Feature'].replace('_', ' ').title()
    print(f" * {clean_feature_name}: accounts for {row['Importance']*100:.1f}% of the model's risk assessment outcome.")


[EXECUTIVE INSIGHT] Core Drivers of Credit Defaults for Business Teams:
 * Loan Percent Income: accounts for 32.7% of the model's risk assessment outcome.
 * Loan Int Rate: accounts for 15.6% of the model's risk assessment outcome.
 * Person Income: accounts for 13.3% of the model's risk assessment outcome.
 * Loan Grade D: accounts for 12.2% of the model's risk assessment outcome.
